In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:


dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
rawpath_patientclassification=ExternalLocation_raw+'/MasterData/PatientClassification/'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
silverpath_promotion=ExternalLocationName_silver+'/DIMPromotion'
silverpath_patientclassification=ExternalLocationName_silver+'/DIMPatientClassification'


In [0]:
# #Declaring Source and Destination Paths
# rawpath_patientclassification = '/mnt/raw/MasterData/PatientClassification/'

# silverpath_promotion = '/mnt/silver/DIMPromotion'
# silverpath_patientclassification = '/mnt/silver/DIMPatientClassification'

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

# PatientClassification

In [0]:
df_Promotion = spark.read.format('delta').load(silverpath_promotion).filter('current_date() >= EffectiveDate and current_date() <= TerminationDate').select('PromotionUUID','PromotionName')

df_patientclassification = spark.read.format("csv").option("header",True).load(rawpath_patientclassification)

df_patientclassification = df_patientclassification.join(df_Promotion,'PromotionName','inner')\
                .withColumn('PatientClassificationUUID',expr('uuid()'))\
                .withColumn('ListPrice',col('ListPrice').cast("Int"))\
                .withColumn('Active',lit('1'))\
                .withColumn('CreatedBy',lit('ADB_PatientClassification'))\
                .withColumn('CreatedDate',current_timestamp())\
                .withColumn('UpdatedBy',lit('ADB_PatientClassification'))\
                .withColumn('UpdatedDate',current_timestamp())\
                .select('PatientClassificationUUID','PromotionUUID','PatientClassificationName','DisplayName','PatientClassificationDescription','SKUCode','ListPrice','PatientClassificationGroupName','InvoiceMultiplier','InvoiceImpactFlag','CreatedDate','EffectiveDate','TerminationDate','CreatedBy','UpdatedDate','UpdatedBy','Active','InvoiceCalculationType','DisplayInd')

df_patientclassification.display()

In [0]:
from pyspark.sql.functions import to_date

# Convert date columns to the correct format
df_patientclassification = df_patientclassification.withColumn(
    "EffectiveDate", to_date("EffectiveDate", "M/d/yyyy")
).withColumn(
    "TerminationDate", to_date("TerminationDate", "M/d/yyyy")
)


targetDF = DeltaTable.forPath(spark, silverpath_patientclassification)  
(targetDF.alias("tgt")
  .merge(
      df_patientclassification.alias("src"), 
      "src.PatientClassificationName = tgt.PatientClassificationName and tgt.PromotionUUID = src.PromotionUUID and tgt.Active = 1 and src.DisplayName = tgt.DisplayName"
  )
  .whenMatchedUpdate(
    set = {
        "tgt.PromotionUUID": "src.PromotionUUID", 
        "tgt.DisplayName": "src.DisplayName",        
        "tgt.PatientClassificationDescription": "src.PatientClassificationDescription",
        "tgt.SKUCode": "src.SKUCode",
        "tgt.ListPrice": "src.ListPrice",
        "tgt.PatientClassificationGroupName": "src.PatientClassificationGroupName",
        "tgt.InvoiceMultiplier": "src.InvoiceMultiplier",
        "tgt.InvoiceImpactFlag": "src.InvoiceImpactFlag",
        "tgt.InvoiceCalculationType": "src.InvoiceCalculationType",
        "tgt.EffectiveDate": "src.EffectiveDate",
        "tgt.TerminationDate": "src.TerminationDate",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.Active": "src.Active",
        "tgt.DisplayInd": "src.DisplayInd"
    }
  )
  .whenNotMatchedInsert(values={
        "tgt.PatientClassificationUUID": "src.PatientClassificationUUID",
        "tgt.PromotionUUID": "src.PromotionUUID",
        "tgt.PatientClassificationName": "src.PatientClassificationName",
        "tgt.DisplayName": "src.DisplayName",
        "tgt.PatientClassificationDescription": "src.PatientClassificationDescription",
        "tgt.SKUCode": "src.SKUCode",
        "tgt.ListPrice": "src.ListPrice",
        "tgt.PatientClassificationGroupName": "src.PatientClassificationGroupName",
        "tgt.InvoiceMultiplier": "src.InvoiceMultiplier",
        "tgt.InvoiceImpactFlag": "src.InvoiceImpactFlag",  
        "tgt.InvoiceCalculationType": "src.InvoiceCalculationType",
        "tgt.EffectiveDate": "src.EffectiveDate",
        "tgt.TerminationDate": "src.TerminationDate",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDate": "src.CreatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.Active": "src.Active",
        "tgt.DisplayInd": "src.DisplayInd"
    })
  .execute()
)

In [0]:
%sql
select * From promotion.dim_patientclassification